In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/events.csv")

In [3]:
df_MLxG = df.copy()
df_MLxG = df_MLxG[["event_pk", "match_id", "player_id", "x", "y", "end_x", "end_y", "goal_mouth_y","goal_mouth_z", "blocked_x", "blocked_y", "type", "is_goal", "is_shot", "qualifiers"]]

In [4]:
df_MLxG["distance"] = np.sqrt((100 - df_MLxG["x"])**2 + (df_MLxG["goal_mouth_y"] -df_MLxG["y"])**2)
df_MLxG = df_MLxG[df_MLxG["is_shot"].notna()].copy()

In [5]:
PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0
goal_width_units = 7.32 / PITCH_WIDTH_M * 100
post1_y, post2_y = 50 - goal_width_units / 2, 50 + goal_width_units / 2

def shot_angle(row):
    dx = 100 - row["x"]
    a = np.arctan2(goal_width_units * dx, dx**2 + (row["y"] - 50)**2 - (goal_width_units / 2)**2)
    return abs(a)

df_MLxG["angle_to_goal"] =df_MLxG.apply(shot_angle, axis=1)

In [6]:
import json

def get_qualifier_tags(qualifiers_raw):
    if not qualifiers_raw or pd.isna(qualifiers_raw):
        return set()
    try:
        qualifiers = json.loads(qualifiers_raw) if isinstance(qualifiers_raw, str) else qualifiers_raw
    except (TypeError, ValueError):
        return set()
    return {q.get("type", {}).get("displayName") for q in qualifiers}

def get_body_part(qualifiers_raw):
    tags = get_qualifier_tags(qualifiers_raw)
    if "Head" in tags:
        return "Head"
    if "RightFoot" in tags or "LeftFoot" in tags:
        return "Foot"
    return "Other"

df_MLxG["body_part"] = df_MLxG["qualifiers"].apply(get_body_part)

In [7]:
def get_play_pattern(qualifiers_raw):
    tags = get_qualifier_tags(qualifiers_raw)
    if "FastBreak" in tags:
        return "FastBreak"
    if "SetPiece" in tags or "FromCorner" in tags:
        return "SetPiece"
    return "RegularPlay"

df_MLxG["play_pattern"] = df_MLxG["qualifiers"].apply(get_play_pattern)

In [8]:
df_MLxG["is_goal"] = (df_MLxG["type"] == "Goal").astype(int)

In [9]:
print(df_MLxG["body_part"].value_counts())
print(df_MLxG["play_pattern"].value_counts())

body_part
Foot     6294
Head     1303
Other      21
Name: count, dtype: int64
play_pattern
RegularPlay    5324
SetPiece       1703
FastBreak       591
Name: count, dtype: int64


In [10]:
df_MLxG = pd.get_dummies(df_MLxG, columns=["body_part","play_pattern"])

In [11]:
print(df_MLxG.columns.tolist())

['event_pk', 'match_id', 'player_id', 'x', 'y', 'end_x', 'end_y', 'goal_mouth_y', 'goal_mouth_z', 'blocked_x', 'blocked_y', 'type', 'is_goal', 'is_shot', 'qualifiers', 'distance', 'angle_to_goal', 'body_part_Foot', 'body_part_Head', 'body_part_Other', 'play_pattern_FastBreak', 'play_pattern_RegularPlay', 'play_pattern_SetPiece']


In [12]:
from sklearn.model_selection import train_test_split
unique_matches = df_MLxG["match_id"].unique()
print(len(unique_matches))

306


In [13]:
train_matches, test_matches = train_test_split(unique_matches, test_size=0.2, random_state=42)
print(len(train_matches), len(test_matches))

244 62


In [14]:
df_train = df_MLxG[df_MLxG["match_id"].isin(train_matches)]
df_test = df_MLxG[df_MLxG["match_id"].isin(test_matches)]
print(len(df_train) + len(df_test) == len(df_MLxG))

True


In [15]:
X_train = df_train[["distance",'angle_to_goal', 'body_part_Foot', 'body_part_Head', 'body_part_Other', 'play_pattern_FastBreak', 'play_pattern_RegularPlay', 'play_pattern_SetPiece']]
Y_train = df_train["is_goal"]
X_test = df_test[["distance",'angle_to_goal', 'body_part_Foot', 'body_part_Head', 'body_part_Other', 'play_pattern_FastBreak', 'play_pattern_RegularPlay', 'play_pattern_SetPiece']]
Y_test = df_test["is_goal"]

In [16]:
from sklearn.linear_model import LogisticRegression


In [17]:
model = LogisticRegression()
model.fit(X_train, Y_train)
probabilities = model.predict_proba(X_test)[:,1]

In [18]:

print(X_train.isna().sum())
print(Y_train.isna().sum())

distance                    0
angle_to_goal               0
body_part_Foot              0
body_part_Head              0
body_part_Other             0
play_pattern_FastBreak      0
play_pattern_RegularPlay    0
play_pattern_SetPiece       0
dtype: int64
0


In [19]:
from sklearn.metrics import log_loss
log_loss(Y_test, probabilities)

0.30871651298928415

In [20]:
from sklearn.metrics import roc_auc_score
roc_auc_score(Y_test, probabilities)

0.7781227413102042

In [21]:
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(Y_test, probabilities)
print(prob_true, prob_pred)

[0.07959479 0.38931298 0.47368421 0.375      0.83333333] [0.08382558 0.25888222 0.4647754  0.70060657 0.88489091]


In [22]:
bins = pd.cut(probabilities, bins=5)
print(bins.value_counts().sort_index())

(0.00879, 0.198]    1379
(0.198, 0.385]       133
(0.385, 0.573]        19
(0.573, 0.761]         9
(0.761, 0.949]         6
Name: count, dtype: int64


In [23]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train, Y_train)

xgb_proba_train = xgb_model.predict_proba(X_train)[:, 1]
xgb_proba_test = xgb_model.predict_proba(X_test)[:, 1]

print("Train log loss:", log_loss(Y_train, xgb_proba_train))
print("Test log loss:", log_loss(Y_test, xgb_proba_test))
print("Test AUC:", roc_auc_score(Y_test, xgb_proba_test))

Train log loss: 0.2601013053144708
Test log loss: 0.2740684129332899
Test AUC: 0.8301034069255535


In [24]:
X_all = df_MLxG[["distance", "angle_to_goal", "body_part_Foot", "body_part_Head", "body_part_Other", "play_pattern_FastBreak", "play_pattern_RegularPlay", "play_pattern_SetPiece"]]
df_MLxG["xg"] = xgb_model.predict_proba(X_all)[:, 1]

In [25]:
print(df_MLxG[["event_pk", "distance", "angle_to_goal", "xg"]].head(10))

     event_pk   distance  angle_to_goal        xg
153       154  13.416408       0.730531  0.168370
240       241  27.485451       0.439282  0.047968
308       309  19.769927       0.149704  0.075519
326       327  22.214410       0.475498  0.090230
347       348  40.603694       0.184695  0.017456
521       522  27.810789       0.302903  0.019579
535       536  17.507141       0.440961  0.122540
544       545   2.088061       0.487168  0.060333
578       579  25.316003       0.399083  0.037053
592       593  13.974262       0.741264  0.161096


In [26]:
df_xg_export = df_MLxG[["event_pk", "xg"]]
df_xg_export.to_csv("../data/processed/staging_xg.csv", index=False)

In [27]:
df_MLxA = df[df["type"] == "Pass"].copy()
df_MLxA["tags"] = df_MLxA["qualifiers"].apply(get_qualifier_tags)
df_MLxA = df_MLxA[df_MLxA["tags"].apply(lambda t: "KeyPass" in t)].copy()

In [28]:
df_shots = df[df["is_shot"].notna()][["event_pk", "match_id", "team_id", "expanded_minute", "second"]].copy()
df_shots = df_shots.merge(df_MLxG[["event_pk", "xg"]], on="event_pk", how="left")
df_shots["clock"] = df_shots["expanded_minute"] * 60 + df_shots["second"]
df_shots = df_shots.sort_values("event_pk")

In [29]:
# related_event_id is unreliable to link a key pass to its shot -> match on
# same match, same team, next shot chronologically, within a short time window
df_MLxA["clock"] = df_MLxA["expanded_minute"] * 60 + df_MLxA["second"]
shots_by_match_team = {key: group for key, group in df_shots.groupby(["match_id", "team_id"])}

def find_resulting_shot_xg(row, window=20):
    candidates = shots_by_match_team.get((row["match_id"], row["team_id"]))
    if candidates is None:
        return np.nan
    candidates = candidates[
        (candidates["event_pk"] > row["event_pk"]) &
        (candidates["clock"] >= row["clock"]) &
        (candidates["clock"] - row["clock"] <= window)
    ]
    if candidates.empty:
        return np.nan
    return candidates.iloc[0]["xg"]

df_MLxA["xa"] = df_MLxA.apply(find_resulting_shot_xg, axis=1)

In [30]:
print(df_MLxA["xa"].notna().sum(), "/", len(df_MLxA))

5348 / 5355


In [31]:
df_xa_player = df_MLxA.groupby(["player_id", "player"])["xa"].sum().reset_index()
df_xa_player = df_xa_player.sort_values("xa", ascending=False)
df_xa_player.head(20)

,player_id,player,xa
75,286443.0,Matthieu Udol,8.935984
17,99143.0,Florian Thauvin,7.247798
28,113747.0,Adrien Thomasson,7.213205
96,301869.0,Jonathan Clauss,6.616831
155,367782.0,Mason Greenwood,6.413670
123,339671.0,Aron Dønnum,6.001986
91,300291.0,Romain Del Castillo,5.927033
89,299562.0,Ludovic Blas,5.524847
401,556058.0,Arsène Kouassi,5.063216
156,367803.0,Giorgi Tsitaishvili,5.054551


In [32]:
df_xa_export = df_MLxA[df_MLxA["xa"].notna()][["event_pk", "xa"]]
df_xa_export.to_csv("../data/processed/staging_xa.csv", index=False)


In [33]:
print(df_MLxA["xa"].notna().sum(), "/", len(df_MLxA))

5348 / 5355


In [34]:
def find_resulting_shot_xg(row, window=20):
    candidates = shots_by_match_team.get((row["match_id"], row["team_id"]))
    if candidates is None:
        return pd.Series([np.nan, np.nan])
    candidates = candidates[
        (candidates["event_pk"] > row["event_pk"]) &
        (candidates["clock"] >= row["clock"]) &
        (candidates["clock"] - row["clock"] <= window)
    ]
    if candidates.empty:
        return pd.Series([np.nan, np.nan])
    best = candidates.iloc[0]
    return pd.Series([best["xg"], best["clock"] - row["clock"]])

df_MLxA[["xa", "gap_seconds"]] = df_MLxA.apply(find_resulting_shot_xg, axis=1)

In [41]:
print(df_MLxA["gap_seconds"].describe())

count    5348.000000
mean        2.861631
std         1.833626
min         0.000000
25%         2.000000
50%         2.000000
75%         4.000000
max        16.000000
Name: gap_seconds, dtype: float64


In [36]:
print(df_xa_player.head(20))

     player_id                  player        xa
75    286443.0           Matthieu Udol  8.935984
17     99143.0         Florian Thauvin  7.247798
28    113747.0        Adrien Thomasson  7.213205
96    301869.0         Jonathan Clauss  6.616831
155   367782.0         Mason Greenwood  6.413670
123   339671.0             Aron Dønnum  6.001986
91    300291.0     Romain Del Castillo  5.927033
89    299562.0            Ludovic Blas  5.524847
401   556058.0          Arsène Kouassi  5.063216
156   367803.0     Giorgi Tsitaishvili  5.054551
331   475462.0  Matias Fernandez-Pardo  5.043496
334   478033.0          Afonso Moreira  4.994387
255   431346.0        Lassine Sinayoko  4.860066
245   426648.0       Maghnes Akliouche  4.646677
154   367780.0             Lee Kang-In  4.622249
76    288883.0            Maxime Lopez  4.504015
225   422186.0             Ilan Kebbal  4.450145
21    101955.0                 Emerson  4.447159
108   322671.0           Gauthier Hein  4.407952
268   438349.0      

In [38]:
df_pms = pd.read_csv("../data/processed/player_match_stats.csv")
xg_per_match = df_MLxG.groupby(["player_id", "match_id"])["xg"].sum().reset_index()
xa_per_match = df_MLxA.groupby(["player_id", "match_id"])["xa"].sum().reset_index()

df_pms_updated = df_pms.merge(xg_per_match, on=["player_id", "match_id"], how="left")
df_pms_updated = df_pms_updated.merge(xa_per_match, on=["player_id", "match_id"], how="left")

df_pms_updated["xg"] = df_pms_updated["xg"].fillna(0)
df_pms_updated["xa"] = df_pms_updated["xa"].fillna(0)

In [40]:
print(df_pms_updated.shape)
print(df_pms_updated[["player_id", "match_id", "xg", "xa"]].head(10))

(9413, 30)
   player_id  match_id        xg   xa
0       6683   1911338  0.000000  0.0
1       6683   1911346  0.067933  0.0
2       6683   1911441  0.000000  0.0
3       6683   1911449  0.033505  0.0
4       6683   1911460  0.094948  0.0
5       6683   1911465  0.000000  0.0
6       6683   1911476  0.000000  0.0
7       6683   1911495  0.000000  0.0
8       6683   1911498  0.000000  0.0
9       6683   1911513  0.023957  0.0


In [43]:
df_staging_xgxa = df_pms_updated[["player_id", "match_id", "xg", "xa"]]
df_staging_xgxa = df_staging_xgxa.rename(columns={"xg": "total_xg", "xa": "total_xa"})
df_staging_xgxa.to_csv("../data/processed/staging_xg_xa_match.csv", index=False)


In [45]:
season_stats = df_pms_updated.groupby("player_id").agg(
    matches_played=("match_id", "count"),
    total_minutes=("minutes_played", "sum"),
    goals=("goals", "sum"),
    own_goals=("own_goals", "sum"),
    assists=("assists", "sum"),
    key_passes=("key_passes", "sum"),
    total_passes=("total_passes", "sum"),
    successful_passes=("successful_passes", "sum"),
    crosses=("crosses", "sum"),
    long_balls=("long_balls", "sum"),
    total_shots=("total_shots", "sum"),
    shots_on_target=("shots_on_target", "sum"),
    shots_off_target=("shots_off_target", "sum"),
    shots_post=("shots_post", "sum"),
    fouls_committed=("fouls_committed", "sum"),
    yellow_cards=("yellow_cards", "sum"),
    red_cards=("red_cards", "sum"),
    offsides=("offsides", "sum"),
    corners_taken=("corners_taken", "sum"),
    gk_claims=("gk_claims", "sum"),
    gk_pickups=("gk_pickups", "sum"),
    gk_punches=("gk_punches", "sum"),
    gk_sweeper_actions=("gk_sweeper_actions", "sum"),
    gk_crosses_not_claimed=("gk_crosses_not_claimed", "sum"),
    goals_conceded=("goals_conceded", "sum"),
    total_xg=("xg", "sum"),
    total_xa=("xa", "sum"),
).reset_index()

int_cols = [c for c in season_stats.columns if c not in ["player_id", "total_xg", "total_xa"]]
for col in int_cols:
    season_stats[col] = season_stats[col].astype("Int64")

season_stats.to_csv("../data/processed/player_season_stats.csv", index=False)
print(season_stats.shape)

(550, 28)


In [46]:
season_stats.columns

Index(['player_id', 'matches_played', 'total_minutes', 'goals', 'own_goals',
       'assists', 'key_passes', 'total_passes', 'successful_passes', 'crosses',
       'long_balls', 'total_shots', 'shots_on_target', 'shots_off_target',
       'shots_post', 'fouls_committed', 'yellow_cards', 'red_cards',
       'offsides', 'corners_taken', 'gk_claims', 'gk_pickups', 'gk_punches',
       'gk_sweeper_actions', 'gk_crosses_not_claimed', 'goals_conceded',
       'total_xg', 'total_xa'],
      dtype='object')